In [1]:
import gymnasium as gym 
import r2l

In [2]:
env = gym.make("FrozenLake-v1", render_mode="ansi") # 字符打印出环境地图
env = env.unwrapped  # 解封装才能访问状态转移矩阵P
obs, info = env.reset()
print(env.render())  # 渲染环境



SFFF
FHFH
FFFH
HFFG



In [3]:
print(env.P) # 里面是一个字典{状态0：{动作1:[p, next_state, r, done]}, 状态1:{...}}

{0: {0: [(0.3333333333333333, 0, 0.0, False), (0.3333333333333333, 0, 0.0, False), (0.3333333333333333, 4, 0.0, False)], 1: [(0.3333333333333333, 0, 0.0, False), (0.3333333333333333, 4, 0.0, False), (0.3333333333333333, 1, 0.0, False)], 2: [(0.3333333333333333, 4, 0.0, False), (0.3333333333333333, 1, 0.0, False), (0.3333333333333333, 0, 0.0, False)], 3: [(0.3333333333333333, 1, 0.0, False), (0.3333333333333333, 0, 0.0, False), (0.3333333333333333, 0, 0.0, False)]}, 1: {0: [(0.3333333333333333, 1, 0.0, False), (0.3333333333333333, 0, 0.0, False), (0.3333333333333333, 5, 0.0, True)], 1: [(0.3333333333333333, 0, 0.0, False), (0.3333333333333333, 5, 0.0, True), (0.3333333333333333, 2, 0.0, False)], 2: [(0.3333333333333333, 5, 0.0, True), (0.3333333333333333, 2, 0.0, False), (0.3333333333333333, 1, 0.0, False)], 3: [(0.3333333333333333, 2, 0.0, False), (0.3333333333333333, 1, 0.0, False), (0.3333333333333333, 0, 0.0, False)]}, 2: {0: [(0.3333333333333333, 2, 0.0, False), (0.3333333333333333

In [4]:
holes = set()
ends = set()

# s_: [p, next_state, r, done]
for s in env.P:
    for a in env.P[s]:
        for s_ in env.P[s][a]:
            if s_[2] == 1.0: # 获取的奖励是1，代表目标|终点
                ends.add(s_[1])
            if s_[3] == True:
                holes.add(s_[1])

holes = holes -ends
print("冰洞索引：", holes)
print("目标索引：", ends)

冰洞索引： {11, 12, 5, 7}
目标索引： {15}


In [5]:
for a in env.P[14]:
    print(env.P[14][a])

[(0.3333333333333333, 10, 0.0, False), (0.3333333333333333, 13, 0.0, False), (0.3333333333333333, 14, 0.0, False)]
[(0.3333333333333333, 13, 0.0, False), (0.3333333333333333, 14, 0.0, False), (0.3333333333333333, 15, 1.0, True)]
[(0.3333333333333333, 14, 0.0, False), (0.3333333333333333, 15, 1.0, True), (0.3333333333333333, 10, 0.0, False)]
[(0.3333333333333333, 15, 1.0, True), (0.3333333333333333, 10, 0.0, False), (0.3333333333333333, 13, 0.0, False)]


在一个位置上有概率随机滑向三个方向，这三个方向是给定的，也可能是留在原地的概率

In [9]:
#策略迭代算法
action_meaning = ['⬅️', '⬇️', '➡️', '⬆️'] # gym 预先设定好的动作含义
theta = 1e-5
gamma = 0.9
agent = r2l.PolicyIteration(env, theta, gamma)
agent.policy_iteration()
r2l.print_agent(agent, action_meaning, [5, 7, 11, 12], [15])

策略评估进行25轮后完成
策略提升完成
策略评估进行58轮后完成
策略提升完成
状态价值：
 0.069  0.061  0.074  0.056 
 0.092  0.000  0.112  0.000 
 0.145  0.247  0.300  0.000 
 0.000  0.380  0.639  0.000 
策略：
⬅️🅾️🅾️🅾️ 🅾️🅾️🅾️⬆️ ⬅️🅾️🅾️🅾️ 🅾️🅾️🅾️⬆️ 
⬅️🅾️🅾️🅾️ ⚠️⚠️⚠️⚠️ ⬅️🅾️➡️🅾️ ⚠️⚠️⚠️⚠️ 
🅾️🅾️🅾️⬆️ 🅾️⬇️🅾️🅾️ ⬅️🅾️🅾️🅾️ ⚠️⚠️⚠️⚠️ 
⚠️⚠️⚠️⚠️ 🅾️🅾️➡️🅾️ 🅾️⬇️🅾️🅾️ ✅✅✅✅ 


In [10]:
# 价值迭代算法
action_meaning = ['⬅️', '⬇️', '➡️', '⬆️'] # gym 预先设定好的动作含义
theta = 1e-5
gamma = 0.9
agent = r2l.ValueIteration(env, theta, gamma)
agent.value_iteration()
r2l.print_agent(agent, action_meaning, [5, 7, 11, 12], [15])

价值迭代一共进行60轮
状态价值：
 0.069  0.061  0.074  0.056 
 0.092  0.000  0.112  0.000 
 0.145  0.247  0.300  0.000 
 0.000  0.380  0.639  0.000 
策略：
⬅️🅾️🅾️🅾️ 🅾️⬇️🅾️🅾️ ⬅️🅾️🅾️🅾️ 🅾️🅾️🅾️⬆️ 
⬅️🅾️🅾️🅾️ ⚠️⚠️⚠️⚠️ ⬅️🅾️🅾️🅾️ ⚠️⚠️⚠️⚠️ 
🅾️⬇️🅾️🅾️ ⬅️🅾️🅾️🅾️ ⬅️🅾️🅾️🅾️ ⚠️⚠️⚠️⚠️ 
⚠️⚠️⚠️⚠️ 🅾️⬇️🅾️🅾️ 🅾️⬇️🅾️🅾️ ✅✅✅✅ 


两个算法的结果一致，互相验证